# 01 — Data Collection

Pull historical prices and macro indicators. Goal: have everything we need in `data/raw/` before moving to analysis.

---

In [ ]:
import sys
sys.path.append('..')  # so we can import from src/

import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

## 1.1 Price Data

In [ ]:
tickers = ['SPY', 'QQQ', 'IWM', 'EFA', 'TLT', 'IEF', 'HYG', 'GLD', '^VIX']

prices = yf.download(tickers, start='2005-01-01', auto_adjust=True, progress=False)['Close']
prices.index = pd.to_datetime(prices.index)

print(f'Shape: {prices.shape}')
print(f'Date range: {prices.index[0].date()} → {prices.index[-1].date()}')
prices.tail()

In [ ]:
# check for missing data
missing = prices.isna().sum()
print('Missing values per ticker:')
print(missing[missing > 0])

In [ ]:
# quick visual
fig, axes = plt.subplots(3, 3, figsize=(15, 9))
fig.patch.set_facecolor('#0f0f0f')

for ax, ticker in zip(axes.flatten(), tickers):
    ax.set_facecolor('#1a1a1a')
    s = prices[ticker].dropna()
    s_norm = s / s.iloc[0]  # normalize to 1
    ax.plot(s_norm.index, s_norm.values, color='#00d4aa', linewidth=0.8)
    ax.set_title(ticker, color='#eee', fontsize=10)
    ax.tick_params(colors='#666', labelsize=7)
    for spine in ax.spines.values():
        spine.set_color('#333')

plt.suptitle('Normalized Prices (2005 = 1.0)', color='#eee', fontsize=13)
plt.tight_layout()
plt.savefig('../reports/figures/01_normalized_prices.png', dpi=150, bbox_inches='tight',
            facecolor='#0f0f0f')
plt.show()

In [ ]:
# save
prices.to_csv('../data/raw/prices.csv')
print('Saved prices.csv')

## 1.2 Macro Indicators (FRED)

- **T10Y2Y**: 10Y - 2Y yield spread (negative = inverted = recession signal)
- **BAMLH0A0HYM2**: High yield credit spread
- **FEDFUNDS**: Fed funds rate

In [ ]:
import pandas_datareader.data as web

fred_series = {
    'yield_curve':   'T10Y2Y',
    'hy_spread':     'BAMLH0A0HYM2',
    'fed_funds':     'FEDFUNDS',
}

macro = pd.DataFrame()
for label, series_id in fred_series.items():
    try:
        s = web.DataReader(series_id, 'fred', '2005-01-01')
        macro[label] = s[series_id]
        print(f'  {label}: OK ({len(s)} obs)')
    except Exception as e:
        print(f'  {label}: FAILED — {e}')

macro.tail()

In [ ]:
macro.to_csv('../data/raw/macro_indicators.csv')
print('Saved macro_indicators.csv')

## Notes

- VIX data from yfinance goes back to 1990 which is good
- FRED data is monthly for some series (fed funds) — will need to handle frequency mismatch when merging with daily prices
- EEM (emerging markets) has some gaps early on — might drop from universe or fill forward

**Next:** `02_eda_and_regime_labeling.ipynb`